In [ ]:
import lyart_spec as las
import numpy as np
import matplotlib.pyplot as plt

## Function and parameter description
There are three main functions for evaluating Lya spectra with RT using closed-form solutions/formulae `lya_spec_cls`, series solutions `lya_spec_ses`, and RT simulations `lya_spec_sim`.

Here is a brief description of the input parameter that all three functions share. 
+ `tau0 (float)`: Optical depth
+ `taus2tau0 (float)`: The ratio of source optical depth to cloud optical depth, between 0.0 and 1.0. 
+ `xi (float)`: Initial frequency.  
+ `geometry (str)`: `"sla"`, `"cyl"`, `"sph"` for slab, cylindrical, and spherical geometry, respectively.  
+ `v2b (float)`: The ratio of cloud edge velocity to thermal velocity.  
+ `recoil_flag (bool)`: `True`: with recoil; `False`: without recoil.  
+ `T (float)`: Temperature of the gas cloud in K.

Here is a brief description of the additional input parameter for `lya_spec_cls`. 

+ `x (numpy.ndarray)`: A 1D array of frequency parameter.

Here is a brief description of the additional input parameters for `lya_spec_ses`. 

+ `x (numpy.ndarray)`: A 1D array of frequency parameter.

+ `Nses (int)`: The number of terms to be evaluated in the series. For Nses=1000, the series solution will be evaluated from n=  0 to n=999, 1000 terms in total. 

## Caution 

For function `lya_spec_cls` of closed-form solutions/formulae, 

+ The closed-form solution/formula for non-zero `v2b` is only valid for `taus2tau0=0` and `xi=0`. Switch to series solutions using function `lya_spec_ses` for non-zero `v2b`, `taus2tau0`, and `xi`. Notice that the series solution for non-zero `v2b` is not accurate for `abs(v2b)>1`. 
+ The closed-form solution/formula for `abs(v2b)>100` is not tested again Monte Carlo radiative transfer simulation. 
+ The closed-form solution/formula for `log10(av*tau0)` outside [2.2, 4.2] is not tested again Monte Carlo radiative transfer simulation. 
+ The recoil correction for non-zero `v2b` is not tested again Monte Carlo radiative transfer simulation. 

For function `lya_spec_ses` of series solutions, 

+ The series solution for `abs(v2b)>1` is no longer accurate. Switch to closed-form solutions/formulae using function `lya_spec_cls` for better accuracy. Notice that the closed-form solution/formula for non-zero `v2b` is only functional for `taus2tau0=0` and `xi=0`. 
+ The recoil correction for non-zero `v2b` is not tested again Monte Carlo radiative transfer simulation.

For function `lya_spec_sim` of simulated spectra, 

+ Only certain discrete values of the input parameters have corresponding RT simulations. When using `lya_spec_sim`, input parameters other than the available ones will raise an error. Check the `README.md` file, Secion **Available parameters of simulated Lya spectra with RT**, for available parameters. 

In [ ]:
# Set the physical parameters. 

# Base
tau0 = 1e5; taus2tau0 = 0.0; xi = 0.0; geometry = "cyl"; v2b = 0.0; recoil_flag = False; T = 10.0

# Effects of cloud optical depth
# tau0 = 1e6; taus2tau0 = 0.0; xi = 0.0; geometry = "cyl"; v2b = 0.0; recoil_flag = False; T = 10.0

# Effects of geometry
# tau0 = 1e5; taus2tau0 = 0.0; xi = 0.0; geometry = "sph"; v2b = 0.0; recoil_flag = False; T = 10.0

# Effects of recoil
# tau0 = 1e5; taus2tau0 = 0.0; xi = 0.0; geometry = "cyl"; v2b = 0.0; recoil_flag = True; T = 10.0

# Effects of source positions w/o recoil, taus2tau0
# tau0 = 1e5; taus2tau0 = 0.9; xi = 0.0; geometry = "cyl"; v2b = 0.0; recoil_flag = False; T = 10.0

# Effects of source positions w/ recoil, taus2tau0
# tau0 = 1e5; taus2tau0 = 0.9; xi = 0.0; geometry = "cyl"; v2b = 0.0; recoil_flag = True; T = 10.0

# Effects of initial frequencies w/o recoil, xi
# tau0 = 1e5; taus2tau0 = 0.0; xi = 8.0; geometry = "cyl"; v2b = 0.0; recoil_flag = False; T = 10.0

# Effects of initial frequencies w/ recoil, xi
# tau0 = 1e5; taus2tau0 = 0.0; xi = 8.0; geometry = "cyl"; v2b = 0.0; recoil_flag = True; T = 10.0

# Effects of velocity gradients, v/b
# tau0 = 1e5; taus2tau0 = 0.0; xi = 0.0; geometry = "cyl"; v2b = 10.0; recoil_flag = False; T = 10.0

# Effects of velocity gradients at T=10^4 K, v/b
# tau0 = 1e5*np.sqrt(1000.0); taus2tau0 = 0.0; xi = 0.0; geometry = "cyl"; v2b = 10**0.6; recoil_flag = False; T = 10000.0

In [ ]:
# The Voigt parameter av depends on temperature T.
av = 4.7e-4/np.sqrt(T/1e4)

# Define the range of frequency parameter x to be evaluated in close-form solutions/formulae and series solutions. 
# If not defined, x will be initialized inside the function. 
xnorm = np.linspace(-10.0, 10.0, 401)
x = xnorm*np.power(av*tau0, 1/3)

# Evaluate LyA spectra using closed-form solutions/formulae. 
xcls, ycls = las.lya_spec_cls(x = x, tau0 = tau0, taus2tau0 = taus2tau0, xi = xi, geometry = geometry, 
                              v2b = v2b, recoil_flag = recoil_flag, T = T)

# Evaluate LyA spectra using series solutions.
xses, yses = las.lya_spec_ses(x = x, tau0 = tau0, taus2tau0 = taus2tau0, xi = xi, geometry = geometry, 
                              v2b = v2b, recoil_flag = recoil_flag, T = T, Nses = 1000)

# Read cached LyA spectra computed from radiative transfer simulations.
# The binning of frequency x is fixed for simulated spectra. 
xsim, ysim = las.lya_spec_sim(tau0 = tau0, taus2tau0 = taus2tau0, xi = xi, geometry = geometry, 
                              v2b = v2b, recoil_flag = recoil_flag, T = T)

In [ ]:
# Plot the spectra from closed-form solutions/formulae, series solutions, and simulations. 
plt.plot(xcls, ycls, linestyle = "-", label = "Closed-form")
plt.plot(xses, yses, linestyle = "dotted", label = "Series")
plt.step(xsim, ysim, label = "RT Sim")
plt.legend(fontsize = 12, frameon = False)
plt.xlabel(r"$x = \Delta \nu/\nu_{\rm D}$", fontsize = 20)
# Factor A is 2 for "sla", 2*pi*r0 for "cyl", and 4*pi*r0**2 for "sph". 
# r0 is the radius of cylinder base/sphere. See Li & Zheng 2026 for details. 
plt.ylabel(r"$J \times A$", fontsize = 20)
plt.xlim(-4*np.power(av*tau0, 1/3), 4*np.power(av*tau0, 1/3))
# plt.ylim(-0.001, 0.01)
plt.tick_params(labelsize =  15)

plt.show()